# MGMT298D: Science and Strategy of AI
## Lab 4: Image Classification with Neural Networks

Welcome to Lab 4! We're now moving into the exciting world of computer vision by building models to classify images of clothing. You'll be working with the **Fashion-MNIST** dataset, a popular benchmark in the machine learning community.

In this lab, you will start with a powerful tree-based model—XGBoost—to establish a strong performance baseline. Then, you will build and train several neural networks of increasing complexity to see how they improve upon this baseline for an image recognition task.

## Learning Objectives
- Compare a powerful tree-based model (XGBoost) with neural networks on an image classification task.
- Prepare image data for modeling, including normalization and splitting.
- Build, visualize, and train neural network models of varying complexity using Keras.
- Implement key techniques like Dropout, Batch Normalization, and Early Stopping.
- Evaluate model performance using accuracy, loss curves, and classification reports.

---

## Setup and Data Preparation

First, we'll import our libraries and load the Fashion-MNIST dataset directly from Keras, which is more efficient than reading from CSV files.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers
from keras.utils import plot_model
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score

# Set random seeds for consistent results
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# IMPORTANT FOR COLAB: To use a GPU, go to Runtime > Change runtime type and select T4 GPU.
# Then, make sure the following line is commented out or removed:
# tf.config.set_visible_devices([], 'GPU')

# Define the class names for our 10 clothing categories
CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

# Load the Fashion-MNIST dataset directly from Keras
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Prepare the data: normalize pixel values to be between 0 and 1 and reshape
x_train_full = x_train_full.astype("float32").reshape(-1, 784) / 255.0
x_test = x_test.astype("float32").reshape(-1, 784) / 255.0

# Create a validation set from the training data for the neural networks
X_train, X_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=SEED
)

print("Data loaded and prepared. ✅")
print(f"Full training data shape: {x_train_full.shape}")
print(f"Test data shape: {x_test.shape}")

### Data Preview

Let's visualize a few examples from the training set to get a feel for the data we're working with.

In [ ]:
# Look at the raw data for the first training example
print(f"Label for one training example: {y_train[6]} ({CLASS_NAMES[y_train[6]]})")

# Reshape the example into a 28x28 matrix to display its pixel values
image_matrix = X_train[6].reshape(28, 28)
df_image = pd.DataFrame(image_matrix)
styled_df = df_image.style.format("{:.2f}").hide()
print("Pixel matrix for the 7th training example (forced 28x28 display):\n")
display(styled_df)

# Visualize the first training example as an image
plt.figure(figsize=(2, 2))
plt.imshow(X_train[6].reshape(28, 28), cmap="gray_r")
plt.title(f"Label: {CLASS_NAMES[y_train[6]]}")
plt.axis("off")
plt.show()

In [ ]:
print("\n---")

# Now, let's visualize the first 10 examples
def show_examples(X, y, n=10):
    plt.figure(figsize=(1.8*n, 2.2))
    for i in range(n):
        plt.subplot(1, n, i+1)
        plt.imshow(X[i].reshape(28,28), cmap="gray_r")
        plt.title(CLASS_NAMES[y[i]])
        plt.axis("off")
    plt.tight_layout()
    plt.show()

print("First 10 images from the training data:")
show_examples(X_train, y_train)

---

## Step 1: Baseline Model - XGBoost Classifier

Before building neural networks, we'll establish a strong baseline using XGBoost, a powerful gradient boosting algorithm. It is highly effective for structured data and will give us a robust benchmark to compare our neural network models against.

In [ ]:
# We will use the full training set for the XGBoost model for a fair comparison
print("Training the baseline XGBoost model...")
# Note: XGBoost will automatically use the GPU in Colab if it's enabled.
xgb_clf = xgb.XGBClassifier(random_state=SEED, n_estimators=10, max_depth=3)
xgb_clf.fit(x_train_full, y_train_full, verbose=True)
print("Training complete. ✅")

# Evaluate the model on the test set
y_pred_xgb = xgb_clf.predict(x_test)
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)

print(f"\nXGBoost - Test Accuracy: {xgb_accuracy*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=CLASS_NAMES))

---

## Step 2: Experimenting with Neural Network Architectures 🧪

Now we'll see if we can beat the XGBoost baseline by using neural networks (Multi-Layer Perceptrons). We will build four models of increasing complexity. To ensure we don't overfit and to make training more efficient, we'll use an **EarlyStopping callback** for all of them.

- **Model 1**: A simple MLP with one hidden layer.
- **Model 2**: A deeper MLP with two hidden layers.
- **Model 3**: A deeper and regularized MLP with three hidden layers and **Dropout**.
- **Model 4**: Our most advanced model, adding **Batch Normalization** for faster, more stable training.

### NN Model 1: Simple MLP (1 Hidden Layer)

Our first neural network will have a single hidden layer with 128 neurons and `ReLU` activation. The final layer uses `softmax` to output probabilities for each of the 10 classes.

In [ ]:
# Define the model architecture
model_1 = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model_1.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

print("NN Model 1 Architecture:")
model_1.summary()

In [ ]:
# Callback for Early Stopping: stops training if validation accuracy doesn't improve for 5 epochs
early_stopping = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)

# Function to plot training history
def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=16)
    # Plot training & validation accuracy values
    ax1.plot(history.history['accuracy'])
    ax1.plot(history.history['val_accuracy'])
    ax1.set_title('Model Accuracy')
    ax1.set_ylabel('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.legend(['Train', 'Validation'], loc='upper left')
    ax1.grid(True)

    # Plot training & validation loss values
    ax2.plot(history.history['loss'])
    ax2.plot(history.history['val_loss'])
    ax2.set_title('Model Loss')
    ax2.set_ylabel('Loss')
    ax2.set_xlabel('Epoch')
    ax2.legend(['Train', 'Validation'], loc='upper left')
    ax2.grid(True)

    plt.show()

# Train the model
print("Training NN Model 1...")
history_1 = model_1.fit(
    X_train, y_train,
    epochs=50, # Set a high number, early stopping will find the best one
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=1
)
print("Training complete. ✅\n")

# Evaluate and plot
test_loss_1, test_acc_1 = model_1.evaluate(x_test, y_test, verbose=0)
print(f"NN Model 1 - Test Accuracy: {test_acc_1*100:.2f}%")
plot_history(history_1, "NN Model 1: Simple MLP")

### NN Model 2: Deeper MLP (2 Hidden Layers)

Next, we'll make the network deeper by adding a second hidden layer. This allows the model to learn features at different levels of abstraction.

In [ ]:
# Define the deeper model architecture
model_2 = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"), # Second hidden layer
    layers.Dense(10, activation="softmax")
])

model_2.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

print("NN Model 2 Architecture:")
model_2.summary()

# Train the model
print("Training NN Model 2...")
history_2 = model_2.fit(
    X_train, y_train,
    epochs=50,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=1
)
print("Training complete. ✅\n")

# Evaluate and plot
test_loss_2, test_acc_2 = model_2.evaluate(x_test, y_test, verbose=0)
print(f"NN Model 2 - Test Accuracy: {test_acc_2*100:.2f}%")
plot_history(history_2, "NN Model 2: Deeper MLP")

### NN Model 3: Deeper & Regularized MLP (3 Hidden Layers + Dropout)

Now we add **Dropout** layers for regularization, which helps prevent overfitting by forcing the network to learn more robust features.

In [ ]:
# Define the deeper, regularized model architecture
model_3 = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3), # Dropout layer
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.3), # Dropout layer
    layers.Dense(8, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model_3.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

print("NN Model 3 Architecture:")
model_3.summary()

# Train the model
print("Training NN Model 3...")
history_3 = model_3.fit(
    X_train, y_train,
    epochs=50,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=1
)
print("Training complete. ✅\n")

# Evaluate and plot
test_loss_3, test_acc_3 = model_3.evaluate(x_test, y_test, verbose=0)
print(f"NN Model 3 - Test Accuracy: {test_acc_3*100:.2f}%")
plot_history(history_3, "NN Model 3: Deeper & Regularized MLP")

### NN Model 4: Advanced MLP (3 Hidden Layers + Dropout + Batch Normalization)

Our final and most advanced model adds **Batch Normalization** layers. This technique helps to accelerate and stabilize training by normalizing the activations of the previous layer, often leading to better performance.

In [ ]:
# Define the advanced model architecture
model_4 = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(32, use_bias=False), # No bias needed before a Batch Norm layer
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),

    layers.Dense(16, use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),

    layers.Dense(8, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model_4.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

print("NN Model 4 Architecture:")
model_4.summary()

# Train the model
print("Training NN Model 4...")
history_4 = model_4.fit(
    X_train, y_train,
    epochs=50,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=1
)
print("Training complete. ✅\n")

# Evaluate and plot
test_loss_4, test_acc_4 = model_4.evaluate(x_test, y_test, verbose=0)
print(f"NN Model 4 - Test Accuracy: {test_acc_4*100:.2f}%")
plot_history(history_4, "NN Model 4: Advanced MLP")

---

## Final Model Comparison and Strategic Implications

Let's summarize the performance of all our models on the test set to determine our champion.

In [ ]:
# Create a summary DataFrame for the final comparison
final_results = pd.DataFrame({
    'Model': [
        'Baseline (XGBoost)',
        'NN Model 1 (1 Hidden Layer)',
        'NN Model 2 (2 Hidden Layers)',
        'NN Model 3 (3 Hidden Layers + Dropout)',
        'NN Model 4 (Advanced MLP)'
    ],
    'Test Accuracy': [
        xgb_accuracy,
        test_acc_1,
        test_acc_2,
        test_acc_3,
        test_acc_4
    ]
})

final_results['Test Accuracy'] = final_results['Test Accuracy'] * 100
final_results = final_results.sort_values(by='Test Accuracy', ascending=False).set_index('Model')

print("=== Final Model Performance Comparison ===")
print(final_results.to_string(formatters={'Test Accuracy': '{:.2f}%'.format}))

### Detailed Metrics Summary

While overall accuracy is a good starting point, looking at metrics like **precision**, **recall**, and **F1-score** gives us a more nuanced view of how each model performs on a per-class basis.

In [ ]:
# Get predictions for the neural network models
y_pred_1 = np.argmax(model_1.predict(x_test), axis=1)
y_pred_2 = np.argmax(model_2.predict(x_test), axis=1)
y_pred_3 = np.argmax(model_3.predict(x_test), axis=1)
y_pred_4 = np.argmax(model_4.predict(x_test), axis=1)

# Create a dictionary of models and their predictions
models = {
    'Baseline (XGBoost)': y_pred_xgb,
    'NN Model 1 (1 Hidden Layer)': y_pred_1,
    'NN Model 2 (2 Hidden Layers)': y_pred_2,
    'NN Model 3 (3 Hidden Layers + Dropout)': y_pred_3,
    'NN Model 4 (Advanced MLP)': y_pred_4
}

# Generate and print a classification report for each model
for model_name, y_pred in models.items():
    print(f"\n--- Classification Report for {model_name} ---")
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

### Strategic Implications

The results clearly show the advantage of neural networks for image data. For an e-commerce company, this increase in accuracy means fewer misclassified items, leading to better inventory management, more reliable search results, and a better customer experience. The investment in a more complex model translates directly to improved operational efficiency and customer satisfaction.

---

## Questions for Analysis

1.  **Baseline vs. NNs**: How did the XGBoost model perform compared to the first neural network (Model 1)? What does this tell you about the relative strengths of these models on image data?

2.  **Impact of Depth**: Compare the performance and training plots of NN Model 1 (1 layer) and NN Model 2 (2 layers). Did adding a layer improve the test accuracy? Did you see any changes in overfitting (i.e., a bigger gap between training and validation accuracy)?

3.  **Impact of Regularization**: Compare NN Model 2 and NN Model 3. Model 3 is deeper and includes Dropout. How did Dropout affect the final test accuracy and the gap between training and validation loss/accuracy? Explain why this might be.

4.  **Impact of Batch Normalization**: Compare NN Model 3 and NN Model 4. The only difference is the addition of Batch Normalization. How did this layer affect the final test accuracy and the training process (e.g., number of epochs before early stopping)?

5.  **Business Recommendation**: Imagine you work for an online retailer. Based on your best model's performance, which clothing category would you be most confident in automatically tagging for inventory management? Which category would you flag for manual review, and why?

---

## Sandbox: Build Your Own Model 🚀

Now it's your turn! Use the cell below to experiment. Try building your own neural network architecture. You can change:
- The number of hidden layers
- The number of neurons in each layer
- The dropout rate
- The activation functions

See if you can build a model that beats the accuracy of Model 4!

In [ ]:
# Your code here!
# 1. Define your model using keras.Sequential

# 2. Compile your model

# 3. Train your model using .fit() (don't forget the early_stopping callback)

# 4. Evaluate your model on the test set

# 5. Plot the training history

---

## Conclusion

In this lab, you have built, trained, and evaluated several neural networks for image classification. You started with a simple baseline and progressively built more complex and effective models by adding hidden layers and introducing regularization. This iterative approach demonstrates a core principle of machine learning engineering: starting simple and methodically experimenting to achieve better performance.